# 💧 LFM2 Inference with vLLM

This notebook demonstrates how to run [LFM2](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda) and [LFM2.5](https://huggingface.co/collections/LiquidAI/lfm25-6839e3e26b2a9fdbde95b341) models using [vLLM](https://github.com/vllm-project/vllm) for high-throughput inference.

**Requires GPU runtime** (`Runtime` → `Change runtime type` → `T4 GPU`).

## Installation

Install vLLM:

In [ ]:
!uv pip install -qqq vllm==0.14

In [ ]:
# These are only needed for Colab
import contextlib
import vllm.utils.system_utils as system_utils

!uv pip install -qqq -U "protobuf<5"
!VLLM_ENABLE_V1_MULTIPROCESSING=0
system_utils.suppress_stdout = contextlib.nullcontext

## Basic Text Generation

Load the model and generate text:

In [ ]:
from vllm import LLM, SamplingParams

# Initialize the model
llm = LLM(model="LiquidAI/LFM2.5-1.2B-Instruct")

# Define sampling parameters
sampling_params = SamplingParams(
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_tokens=512
)

# Generate answer
messages = [{"role": "user", "content": "What is C. elegans?"}]
output = llm.chat(messages, sampling_params)
print(output[0].outputs[0].text)

## Batched Generation

vLLM efficiently processes multiple prompts in a single batch:

In [ ]:
prompts = [
    "Explain quantum computing in one sentence.",
    "What are the benefits of exercise?",
    "Write a haiku about programming.",
]

# Convert to message format for chat API
conversations = [[{"role": "user", "content": p}] for p in prompts]
outputs = llm.chat(conversations, sampling_params)

for i, output in enumerate(outputs):
    print(f"Prompt {i + 1}: {prompts[i]}")
    print(f"Response: {output.outputs[0].text}\n")

## Vision Models

For LFM2-VL vision models, you need a specific vLLM version with vision support:

In [ ]:
# Install vLLM with vision model support (Note: you may need to restart your runtime before installing a new version of vllm)
!VLLM_PRECOMPILED_WHEEL_COMMIT=72506c98349d6bcd32b4e33eec7b5513453c1502 VLLM_USE_PRECOMPILED=1 
!uv pip install -qqq git+https://github.com/vllm-project/vllm.git
!uv pip install -qqq "transformers>=5.0.0" pillow

In [ ]:
# These are only needed for Colab
import contextlib
import vllm.utils.system_utils as system_utils

!VLLM_ENABLE_V1_MULTIPROCESSING=0
!uv pip install -qqq -U "protobuf<5"
system_utils.suppress_stdout = contextlib.nullcontext

In [ ]:
from vllm import LLM, SamplingParams

def build_messages(parts):
    content = []
    for item in parts:
        if item["type"] == "text":
            content.append({"type": "text", "text": item["value"]})
        elif item["type"] == "image":
            content.append({"type": "image_url", "image_url": {"url": item["value"]}})
    return [{"role": "user", "content": content}]

IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"

llm = LLM(
    model="LiquidAI/LFM2.5-VL-1.6B",
    max_model_len=1024,
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
)

# Batch multiple prompts - text-only and multimodal
prompts = [
    [{"type": "text", "value": "What is C. elegans?"}],
    [{"type": "text", "value": "Say hi in JSON format"}],
    [
        {"type": "image", "value": IMAGE_URL},
        {"type": "text", "value": "Describe what you see in this image."},
    ],
]

conversations = [build_messages(p) for p in prompts]
outputs = llm.chat(conversations, sampling_params)

for output in outputs:
    print(output.outputs[0].text)
    print("---")

## Resources

- [LFM2 Documentation](https://docs.liquid.ai/docs/inference/vllm)
- [LFM2 Models on Hugging Face](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda)
- [vLLM Documentation](https://docs.vllm.ai/)